# System Design, Plainly — Part 7: Consistent Hashing (companion notebook)

Every number in the post is recomputed here from scratch. The ring is built with
SHA-256 hashes (any good hash works — the properties are structural, not hash-specific).
Run all cells top to bottom; every cell ends with asserts.

In [ ]:
import hashlib, bisect, random, time

def h64(s: str) -> int:
    return int.from_bytes(hashlib.sha256(s.encode()).digest()[:8], "little")

KEYS = [f"user:{i}" for i in range(100_000)]
key_hash = [h64(k) for k in KEYS]
print(f"{len(KEYS):,} keys hashed; ring space is 2^64")


In [ ]:
# Cell 1 — modulo vs the ring: how many keys move when the cluster grows?
def modulo_moved(n0, n1):
    return sum(1 for kh in key_hash if kh % n0 != kh % n1) / len(KEYS) * 100

def ring_points(n, vnodes=150):
    pts = sorted((h64(f"n{node}v{v}"), node)
                 for node in range(n) for v in range(vnodes))
    return [p[0] for p in pts], [p[1] for p in pts]

def ring_moved(n0, n1, vnodes=150):
    h0, o0 = ring_points(n0, vnodes)
    h1, o1 = ring_points(n1, vnodes)
    moved = 0
    for kh in key_hash:
        if o0[bisect.bisect_left(h0, kh) % len(h0)] != o1[bisect.bisect_left(h1, kh) % len(h1)]:
            moved += 1
    return moved / len(KEYS) * 100

for a, b in [(8, 9), (8, 10), (10, 11)]:
    print(f"{a} -> {b} nodes: modulo moves {modulo_moved(a, b):.1f}% | ring moves {ring_moved(a, b):.1f}%")

assert modulo_moved(8, 9) > 80, "modulo should remap ~everything"
assert ring_moved(8, 9) < 15, "ring should move ~1/N"
assert ring_moved(8, 9) < modulo_moved(8, 9) / 5
print("OK: adding a node moves ~1/N of keys on the ring, ~all of them under modulo")


In [ ]:
# Cell 2 — virtual nodes: how even is the load?
def shares(n_nodes, vnodes, n_keys=100_000):
    hs, owners = ring_points(n_nodes, vnodes)
    counts = [0] * n_nodes
    for kh in key_hash[:n_keys]:
        counts[owners[bisect.bisect_left(hs, kh) % len(hs)]] += 1
    return [c / n_keys * 100 for c in counts]

for v in [1, 10, 50, 150]:
    s = shares(8, v)
    print(f"{v:>3} vnodes/node: busiest {max(s):5.1f}%  quietest {min(s):5.1f}%  (ideal 12.5%)")

assert max(shares(8, 1)) > 40, "1 vnode leaves one node holding ~half the ring"
assert max(shares(8, 150)) < 17, "150 vnodes brings the busiest near ideal"
assert min(shares(8, 150)) > 10
print("OK: vnodes are what make the ring usable — 100-200 per node is the usual range")


In [ ]:
# Cell 3 — the 3 AM test: one node dies. Who absorbs its keys?
def loads(n_nodes, vnodes, dead=None):
    nodes = [n for n in range(n_nodes) if n != dead]
    hs, owners = ring_points(len(nodes), vnodes) if False else (None, None)
    # rebuild the ring over the surviving node ids (vnodes keep their node ids)
    pts = sorted((h64(f"n{node}v{v}"), node)
                 for node in nodes for v in range(vnodes))
    hs = [p[0] for p in pts]; owners = [p[1] for p in pts]
    counts = {n: 0 for n in nodes}
    for kh in key_hash:
        counts[owners[bisect.bisect_left(hs, kh) % len(hs)]] += 1
    return {n: c / len(KEYS) * 100 for n, c in counts.items()}

before1, after1 = loads(8, 1), loads(8, 1, dead=3)
before150, after150 = loads(8, 150), loads(8, 150, dead=3)
succ = max(after1, key=lambda n: after1[n] - before1[n])
print(f"1 vnode:   node 3 held {before1[3]:.1f}%; its successor node {succ} went "
      f"{before1[succ]:.1f}% -> {after1[succ]:.1f}%")
print(f"150 vnodes: node 3 held {before150[3]:.1f}%; biggest survivor gain "
      f"{max(after150[n] - before150[n] for n in after150):.1f} points")

assert after1[succ] - before1[succ] > 10, "without vnodes one node eats the whole arc"
assert max(after150[n] - before150[n] for n in after150) < 3, "with vnodes the load scatters"
print("OK: vnodes turn a cascade into a rounding error")


In [ ]:
# Cell 4 — lookup cost: ring vs Jump vs rendezvous (HRW), measured on this machine
def jump_hash(key, num_buckets):
    b, j = -1, 0
    k = key
    while j < num_buckets:
        b = j
        k = (k * 2862933555777941757 + 1) & 0xFFFFFFFFFFFFFFFF
        j = int((b + 1) * (1 << 31) / ((k >> 33) + 1))
    return b

def bench(fn, n, reps=2000):
    fn(0, n)  # warm up (build cached tables) — not timed
    t0 = time.perf_counter()
    for i in range(reps):
        fn(i, n)
    return (time.perf_counter() - t0) / reps * 1e6

def ring_lookup(i, n, cache={}):
    if n not in cache:
        cache[n] = sorted(h64(f"n{node}v{v}") for node in range(n) for v in range(150))
    return bisect.bisect_left(cache[n], h64(f"key:{i}")) % len(cache[n])

def jump_lookup(i, n):
    return jump_hash(h64(f"key:{i}"), n)

def hrw_lookup(i, n):
    kh = h64(f"key:{i}")
    return max(range(n), key=lambda node: h64(f"{kh}:{node}"))

for n in [100, 1000, 5000]:
    r, j, h = bench(ring_lookup, n), bench(jump_lookup, n), bench(hrw_lookup, n, reps=200)
    print(f"N={n:>5}: ring {r:6.2f} us | jump {j:5.2f} us | rendezvous {h:8.1f} us ({h / r:,.0f}x ring)")

assert bench(hrw_lookup, 1000, reps=200) > 100 * bench(ring_lookup, 1000)
print("OK: rendezvous is O(N) per lookup — fine for 10 nodes, fatal for 5,000")


In [ ]:
# Cell 5 — a working ring in 30 lines: add a node, remove a node, watch 1/N move
class Ring:
    def __init__(self, vnodes=150):
        self.vnodes = vnodes
        self.pts = []  # sorted list of (hash, node)

    def add(self, node):
        self.pts += [(h64(f"{node}#{v}"), node) for v in range(self.vnodes)]
        self.pts.sort()

    def remove(self, node):
        self.pts = [(h, n) for h, n in self.pts if n != node]

    def get(self, key):
        hs = [p[0] for p in self.pts]
        return self.pts[bisect.bisect_left(hs, h64(key)) % len(self.pts)][1]

ring = Ring()
for n in range(8):
    ring.add(f"cache-{n}")
moved = sum(1 for k in KEYS if ring.get(k) != "cache-0")  # sanity: keys spread out
owners = {ring.get(k) for k in KEYS[:1000]}
assert len(owners) == 8, "all 8 nodes should own keys"
before = {k: ring.get(k) for k in KEYS}
ring.add("cache-8")
moved = sum(1 for k in KEYS if ring.get(k) != before[k]) / len(KEYS) * 100
print(f"adding cache-8 moved {moved:.1f}% of keys (ideal ~11.1%)")
assert 8 < moved < 15
ring.remove("cache-8")
assert all(ring.get(k) == before[k] for k in KEYS), "removing it must restore the old mapping exactly"
print("OK: the ring is stable — add/remove only touches the keys that must move")
